# Analyze per Die Matrix

This notebook demonstrates how to find the **reference behavior** for each die matrix, compare individual pieces against it, and identify which process segments show the most variability.

All analysis is performed on the gold parquet dataset (clean, validated pieces with partial times).

In [1]:
# TODO: implement this cell
import pandas as pd
from pathlib import Path

gold_path = Path("../data/gold/pieces.parquet")
gold = pd.read_parquet(gold_path)

gold.head()


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,partial_furnace_to_2nd_s,partial_2nd_to_3rd_s,partial_3rd_to_4th_s,partial_4th_to_aux_s,oee_cycle_time_s,production_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,17.900000,6.700001,13.400000,16.599998,NaN,False,0
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,17.900000,6.700001,13.300001,16.899998,NaN,False,0
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,18.200001,6.599998,13.500000,17.000000,NaN,False,0
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,18.400000,6.700001,13.300001,17.099998,NaN,False,0
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,18.200001,6.599998,13.400002,17.299999,NaN,False,0


## 1. Reference profile per die matrix

The **reference (optimal) behavior** for each die matrix is defined by its median cumulative times. The median is more robust than the mean — it is not pulled by residual edge cases.

This table shows how long a typical piece takes to reach each stage, per matrix.

In [3]:
gold["partial_furnace_to_2nd_s"] = gold["lifetime_2nd_strike_s"]
gold["partial_2nd_to_3rd_s"] = gold["lifetime_3rd_strike_s"] - gold["lifetime_2nd_strike_s"]
gold["partial_3rd_to_4th_s"] = gold["lifetime_4th_strike_s"] - gold["lifetime_3rd_strike_s"]
gold["partial_4th_to_aux_s"] = gold["lifetime_auxiliary_press_s"] - gold["lifetime_4th_strike_s"]
gold["partial_aux_to_bath_s"] = gold["lifetime_bath_s"] - gold["lifetime_auxiliary_press_s"]


In [4]:
# TODO: implement this cell
cumulative_cols = [
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
]

partial_cols = [
    "partial_furnace_to_2nd_s",
    "partial_2nd_to_3rd_s",
    "partial_3rd_to_4th_s",
    "partial_4th_to_aux_s",
    "partial_aux_to_bath_s",
]

reference = gold.groupby("die_matrix")[cumulative_cols + partial_cols].median().round(2)
reference


,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,partial_furnace_to_2nd_s,partial_2nd_to_3rd_s,partial_3rd_to_4th_s,partial_4th_to_aux_s,partial_aux_to_bath_s
die_matrix,,,,,,,,,,
4974,17.3,23.9,37.1,54.2,56.0,17.3,6.5,13.1,17.0,1.8
5052,18.3,25.3,39.3,56.7,58.3,18.3,6.9,13.7,17.3,1.6
5090,17.7,24.6,38.5,56.5,58.1,17.7,6.8,13.8,17.7,1.6
5091,17.9,24.6,38.2,55.2,56.8,17.9,6.6,13.5,17.0,1.6


## 2. Reference partial times per die matrix

The partial times (time spent **between** consecutive stages) are more diagnostically useful than cumulative times — they isolate each segment of the process.

In [5]:
# TODO: implement this cell
partial_cols = [
    "partial_furnace_to_2nd_s",
    "partial_2nd_to_3rd_s",
    "partial_3rd_to_4th_s",
    "partial_4th_to_aux_s",
    "partial_aux_to_bath_s",
]

ref_partials = gold.groupby("die_matrix")[partial_cols].median().round(2)
ref_partials

,partial_furnace_to_2nd_s,partial_2nd_to_3rd_s,partial_3rd_to_4th_s,partial_4th_to_aux_s,partial_aux_to_bath_s
die_matrix,,,,,
4974,17.3,6.5,13.1,17.0,1.8
5052,18.3,6.9,13.7,17.3,1.6
5090,17.7,6.8,13.8,17.7,1.6
5091,17.9,6.6,13.5,17.0,1.6


## 3. Variability per segment per die matrix

Which segments are most variable? High standard deviation relative to the median (coefficient of variation) indicates segments where the process is less stable — these are the primary candidates for delay investigation.

**CV = std / median × 100%** — higher CV means more variability relative to the typical value.

In [6]:
# TODO: implement this cell
cv = gold.groupby("die_matrix")[partial_cols].agg(["median", "std"])
cv_ratio = (cv.xs("std", level=1, axis=1) / cv.xs("median", level=1, axis=1)).round(3)
cv_ratio


,partial_furnace_to_2nd_s,partial_2nd_to_3rd_s,partial_3rd_to_4th_s,partial_4th_to_aux_s,partial_aux_to_bath_s
die_matrix,,,,,
4974,0.091,0.041,0.024,0.052,0.027
5052,0.104,0.073,0.062,0.066,0.031
5090,0.136,0.109,0.081,0.091,0.054
5091,0.137,0.114,0.091,0.061,0.071


## 4. Deviation from reference per piece

For each piece, compute the deviation from its die matrix reference at each stage. Positive deviation = slower than reference. This allows identifying both slow individual pieces and systematic drifts.

In [7]:
# TODO: implement this cell
# Reference medians per matrix
ref = gold.groupby("die_matrix")[partial_cols].median()

# Compute deviations for each piece
deviations = gold[["die_matrix"] + partial_cols].copy()
for col in partial_cols:
    deviations[col + "_dev"] = deviations[col] - deviations["die_matrix"].map(ref[col])

deviations.head()



,die_matrix,partial_furnace_to_2nd_s,partial_2nd_to_3rd_s,partial_3rd_to_4th_s,partial_4th_to_aux_s,partial_aux_to_bath_s,partial_furnace_to_2nd_s_dev,partial_2nd_to_3rd_s_dev,partial_3rd_to_4th_s_dev,partial_4th_to_aux_s_dev,partial_aux_to_bath_s_dev
0,5052,17.900000,6.700001,13.400000,16.599998,1.600002,-0.400000,-0.199999,-0.300001,-0.700001,0.000000
1,5052,17.900000,6.700001,13.300001,16.899998,1.600002,-0.400000,-0.199999,-0.400000,-0.400002,0.000000
2,5052,18.200001,6.599998,13.500000,17.000000,1.600002,-0.099998,-0.300001,-0.200001,-0.299999,0.000000
3,5052,18.400000,6.700001,13.300001,17.099998,1.599998,0.100000,-0.199999,-0.400000,-0.200001,-0.000004
4,5052,18.200001,6.599998,13.400002,17.299999,1.700001,-0.099998,-0.300001,-0.299999,0.000000,0.099998


## 5. Identify slow pieces and their penalized segment

A piece is considered **slow** if its total bath time exceeds the 90th percentile for its die matrix. For each slow piece, identify which segment contributed the most delay.

In [8]:
# TODO: implement this cell
# 90th percentile bath time per matrix
p90 = gold.groupby("die_matrix")["lifetime_bath_s"].quantile(0.90)

# Flag slow pieces
gold["is_slow"] = gold.apply(lambda r: r["lifetime_bath_s"] > p90[r["die_matrix"]], axis=1)

# For slow pieces, find which segment has max positive deviation
dev_cols = [c + "_dev" for c in partial_cols]
slow = deviations.copy()
slow["is_slow"] = gold["is_slow"].values

slow_pieces = slow[slow["is_slow"]].copy()
slow_pieces["penalized_segment"] = slow_pieces[dev_cols].idxmax(axis=1)

slow_pieces[["die_matrix"] + dev_cols + ["penalized_segment"]].head()


,die_matrix,partial_furnace_to_2nd_s_dev,partial_2nd_to_3rd_s_dev,partial_3rd_to_4th_s_dev,partial_4th_to_aux_s_dev,partial_aux_to_bath_s_dev,penalized_segment
114,5052,-1.000000,3.100000,-0.400002,7.500004,0.099995,partial_4th_to_aux_s_dev
225,5052,-1.000000,0.400002,-0.200003,6.700001,0.000000,partial_4th_to_aux_s_dev
235,5052,0.300001,-0.200001,-0.299999,7.099998,-0.000004,partial_4th_to_aux_s_dev
938,5052,-0.400000,-0.299999,-0.200001,10.100002,0.099995,partial_4th_to_aux_s_dev
956,5052,-0.199999,4.400000,-0.100000,0.000000,0.099998,partial_2nd_to_3rd_s_dev


## 6. Slow pieces per die matrix

How slow pieces distribute across die matrices, and which segments are most often penalized per matrix.

In [9]:
# Count slow pieces per die matrix
slow_counts = slow_pieces.groupby("die_matrix").size().rename("slow_count")
total_counts = gold.groupby("die_matrix").size().rename("total_count")

slow_summary = pd.concat([total_counts, slow_counts], axis=1)
slow_summary["slow_pct"] = (slow_summary["slow_count"] / slow_summary["total_count"] * 100).round(2)

print("Slow pieces per die matrix:")
print(slow_summary)

# Most penalized segment per matrix
penalized_by_matrix = (
    slow_pieces.groupby(["die_matrix", "penalized_segment"])
    .size()
    .rename("count")
    .reset_index()
)

# For each matrix, show the top penalized segment
top_penalized = (
    penalized_by_matrix.sort_values(["die_matrix", "count"], ascending=[True, False])
    .groupby("die_matrix")
    .head(1)
)

print("\nMost penalized segment per matrix:")
print(top_penalized.to_string(index=False))




Slow pieces per die matrix:
            total_count  slow_count  slow_pct
die_matrix                                   
4974              15531        1527      9.83
5052              20887        2081      9.96
5090              81677        7946      9.73
5091              22309        2204      9.88

Most penalized segment per matrix:
 die_matrix            penalized_segment  count
       4974 partial_furnace_to_2nd_s_dev   1328
       5052 partial_furnace_to_2nd_s_dev   1708
       5090 partial_furnace_to_2nd_s_dev   5798
       5091 partial_furnace_to_2nd_s_dev   1614


## 7. Time evolution — detecting drift

Does the process get slower over time for a given matrix? Plot the daily median bath time per matrix to detect progressive deterioration.

In [10]:
# TODO: implement this cell
gold["day"] = gold["timestamp"].dt.date

daily = (
    gold.groupby(["die_matrix", "day"])["lifetime_bath_s"]
    .median()
    .reset_index()
    .sort_values(["die_matrix", "day"])
)

# Compare first 5 days vs last 5 days per matrix
drift = []
for m, grp in daily.groupby("die_matrix"):
    first = grp.head(5)["lifetime_bath_s"].median()
    last = grp.tail(5)["lifetime_bath_s"].median()
    drift.append({"die_matrix": m, "first_days_median": first, "last_days_median": last, "delta": last - first})

drift_df = pd.DataFrame(drift).round(3)
drift_df


,die_matrix,first_days_median,last_days_median,delta
0,4974,56.6,55.9,-0.7
1,5052,56.9,59.5,2.6
2,5090,55.8,60.2,4.4
3,5091,56.2,59.7,3.5


## 8. Segment variability ranking

Across all matrices, which process segment is the most unstable? This is where maintenance and process engineering should focus attention.

In [11]:
# TODO: implement this cell
# Compute CV per matrix per segment (from earlier)
cv = gold.groupby("die_matrix")[partial_cols].agg(["median", "std"])
cv_ratio = (cv.xs("std", level=1, axis=1) / cv.xs("median", level=1, axis=1))

# Average CV across matrices to rank segments
segment_cv_rank = cv_ratio.mean().sort_values(ascending=False).round(3)
segment_cv_rank


partial_furnace_to_2nd_s    0.117
partial_2nd_to_3rd_s        0.084
partial_4th_to_aux_s        0.068
partial_3rd_to_4th_s        0.064
partial_aux_to_bath_s       0.046
dtype: float64

## 9. Summary

Key findings from the per-matrix analysis.

In [12]:
# TODO: implement this cell
summary = {
    "total_pieces": len(gold),
    "matrices": gold["die_matrix"].nunique(),
    "slow_pieces_total": gold["is_slow"].sum(),
    "slow_pieces_pct": round(gold["is_slow"].mean() * 100, 2),
}

summary


{'total_pieces': 140404,
 'matrices': 4,
 'slow_pieces_total': np.int64(13758),
 'slow_pieces_pct': np.float64(9.8)}

In [13]:
top_penalized_overall = slow_pieces["penalized_segment"].value_counts().head(3)
top_penalized_overall


penalized_segment
partial_furnace_to_2nd_s_dev    10448
partial_3rd_to_4th_s_dev         1727
partial_4th_to_aux_s_dev         1095
Name: count, dtype: int64